# Stage A — Case-study confirmation (§8.1.6)

Three documented Vietnam aerosol events from the **held-out window**
**(Jan 2025 – Apr 2026)** confirm the merged Stage A product's behaviour on
real episodes (Ahn 2021 §3.3.4 / Gupta 2024 §4.4 analogue):

| Event | Selection criterion | What it tests | Pass |
|---|---|---|---|
| 1 — Severe Hanoi haze | north + dry + PM2.5 > 100 µg/m³ + AOD_merged > 1.0 | merged retains high-AOD events in the high-VZA north (§1 finding 1) | \|Spearman r\| > 0.3 |
| 2 — March-April biomass burning | month ∈ {3, 4} + PM2.5 > 50 µg/m³ | spatial coherence across the 16°N regional-mask boundary | \|Spearman r\| > 0.3 |
| 3 — Precipitation washout | wet season + day-over-day PM2.5 drop ≥ 30 µg/m³ | merged AOD (column) does NOT track surface wet scavenging (negative control) | \|Spearman r\| > 0.3 |

PM2.5 ground truth uses 27 Envisoft stations (10/8/9 north/central/south).
Per §10 #16, completeness over the held-out window alone is relaxed to ≥ 50%
(no station meets the ≥ 85% bar over 510 days).

PM2.5 vs AOD_phys (Stage A3 physics-corrected column) is the primary lens —
`AOD_phys = AOD_merged × (1−RH/100)^0.6 / PBLH` collapses the column AOD to a
surface-concentration proxy that should track PM2.5 directly.  We report
correlations against both `AOD_merged` and `AOD_phys` for comparison.


## 0. Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from config import AERONET_SITES, TEST_START, TEST_END
from validate import (
    extract_aod_pm25_pairs, pm25_case_studies, pm25_coupling_metrics,
    STATIONS_META, PM25_DIR, PM25_COMPLETENESS_MIN,
)

start, end = TEST_START, TEST_END
print(f'Held-out window     : {start} → {end}')
print(f'Station completeness: ≥ {PM25_COMPLETENESS_MIN:.0%} (relaxed per §10 #16)')
print(f'PM2.5 dir           : {PM25_DIR}')
print(f'Station metadata    : {STATIONS_META}')


Held-out window     : 2025-01-01 → 2026-04-30
Station completeness: ≥ 20% (relaxed per §10 #16)
PM2.5 dir           : /home/slow_data/Air_Quality/historical_full_v2
Station metadata    : /home/work1/projects/Air_Quality/Masterdata/envisoft_station_map.csv


## 1. Build daily (AOD, PM2.5) pairs over the held-out window

`extract_aod_pm25_pairs` aggregates all 30-min merged slots in `[start, end]`
to daily means at every Envisoft station's 0.05° cell (mean `AOD_merged` and
mean `AOD_phys_corrected`), then merges with the daily Envisoft PM2.5 mean.
Stations with < 50% completeness over the window are dropped.


In [2]:
pairs = extract_aod_pm25_pairs(start, end)
if pairs.empty:
    print('No daily pairs produced.  Confirm merged NetCDFs cover the window '
          'and that station metadata + PM2.5 CSVs are accessible.')
else:
    print(f'  {len(pairs):,} station-days across '
          f'{pairs["station_name"].nunique()} stations')
    print()
    print('Per-region station counts:')
    print(pairs.groupby('region')['station_name'].nunique().to_string())
pairs.head()


PM2.5 AOD scan: 100%|████████████████████████| 485/485 [03:12<00:00,  2.53day/s]

[extract_aod_pm25_pairs] 111 stations had pairs; 36 pass completeness ≥ 20% (≥ 97 of 485 days).
  5,631 station-days across 36 stations

Per-region station counts:
region
central     9
north      16
south      11


,date,station_name,region,season,month,aod_merged_daily,aod_phys_daily,pm25_daily,n_aod_slots,n_pm25_obs,confidence_flag_mode
0,2025-01-01,Phú Thọ: đường Hùng Vương - Tp Việt Trì (KK),north,dry,1,1.146075,0.001768,79.992442,6,12,2
1,2025-01-01,Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP...,north,dry,1,1.083604,0.001006,133.496083,6,24,2
2,2025-01-01,Hà Nội: ĐHBK cổng Parabol đường Giải Phóng (KK),north,dry,1,1.306576,0.001039,164.125487,10,24,1
3,2025-01-01,Hà Nội: Công viên Nhân Chính - Khuất Duy Tiến ...,north,dry,1,1.301178,0.001058,99.943500,9,10,1
4,2025-01-01,Hưng Yên: số 437 Nguyễn Văn Linh - Tp Hưng Yên...,north,dry,1,1.042197,0.000903,171.414533,9,24,2


## 2. §8.1.6 case-study Spearman r — three events

`pm25_case_studies` selects the three episode windows directly from the
paired DataFrame using the plan's criteria, then for each station × event
computes Spearman r between `AOD_phys_daily` and `PM2.5_daily`.  The
table below filters down to events 1, 2, 4 (the v3.3.2 §8.1.6 set —
event 3 in `pm25_case_studies` is monsoon gap-fill stress, which is a
Stage B test).


In [3]:
if pairs.empty:
    cases = pd.DataFrame()
else:
    cases = pm25_case_studies(pairs)

if cases.empty:
    print('No case-study matches found.')
else:
    keep = ('severe_hanoi_haze', 'biomass_burning_MarApr', 'precip_washout')
    cases = cases[cases['episode_type'].isin(keep)].copy()
    cols  = ['episode_type', 'station_name', 'region', 'N_days',
             'spearman_r', 'spearman_p', 'pearson_r',
             'peak_aod_merged', 'peak_pm25', 'mean_aod_phys', 'mean_pm25']
    cases[cols].sort_values(['episode_type', 'region', 'station_name']).round(3)


### 2a. Pass criterion — `|Spearman r| > 0.3`

Per the plan §8.1.6, an episode passes if `|Spearman r| > 0.3` for the
majority of stations involved.  Report per-event:

* `n_stations` — how many stations matched the episode criterion
* `mean_r` — mean Spearman across those stations
* `pass_frac` — fraction with `|r| > 0.3`
* verdict


In [4]:
if cases.empty:
    print('No cases to summarise.')
else:
    rows = []
    for ep, grp in cases.groupby('episode_type'):
        n   = len(grp)
        mr  = float(grp['spearman_r'].mean())
        pf  = float((grp['spearman_r'].abs() > 0.3).mean())
        rows.append({
            'episode':    ep,
            'n_stations': n,
            'mean_r':     mr,
            'pass_frac':  pf,
            'verdict':    'PASS' if pf >= 0.5 else 'FAIL',
        })
    summary = pd.DataFrame(rows).set_index('episode')
    summary.round(3)


## 3. Time-series figures per event

For each event, plot daily AOD_phys and daily PM2.5 alongside each other (left
y-axis: PM2.5 µg/m³; right y-axis: AOD_phys × 1e4).  One subplot per station
to keep visual scales honest.  The pass criterion (Spearman r > 0.3 for the
majority of stations) sets the qualitative bar, not the visual one.


In [5]:
# def _plot_event(ep_name: str, station_rows: pd.DataFrame, df_pairs: pd.DataFrame):
#     stations = station_rows['station_name'].unique()
#     if len(stations) == 0:
#         print(f'{ep_name}: no qualifying stations.')
#         return
#     ncols = min(3, len(stations))
#     nrows = (len(stations) + ncols - 1) // ncols
#     fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows),
#                              squeeze=False)
#     for ax, sn in zip(axes.flat, stations):
#         sub = df_pairs[df_pairs['station_name'] == sn].sort_values('date')
#         if sub.empty:
#             ax.axis('off'); continue
#         ax2 = ax.twinx()
#         l1, = ax.plot(sub['date'], sub['pm25_daily'], color='C3', marker='.', lw=1, label='PM2.5')
#         l2, = ax2.plot(sub['date'], sub['aod_phys_daily'] * 1e4, color='C0',
#                         marker='.', lw=1, label='AOD_phys×1e4')
#         ax.set_ylabel('PM2.5 (µg/m³)', color='C3'); ax2.set_ylabel('AOD_phys × 1e4', color='C0')
#         r = float(station_rows[station_rows['station_name'] == sn]['spearman_r'].iloc[0])
#         ax.set_title(f'{sn}  (Spearman r = {r:+.2f})', fontsize=9)
#         ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3)
#         ax.legend([l1, l2], ['PM2.5', 'AOD_phys×1e4'], fontsize=8, loc='upper right')
#     for ax in axes.flat[len(stations):]:
#         ax.axis('off')
#     fig.suptitle(ep_name, y=1.02, fontsize=12); fig.tight_layout(); plt.show()

# if not cases.empty:
#     for ep in ('severe_hanoi_haze', 'biomass_burning_MarApr', 'precip_washout'):
#         rows = cases[cases['episode_type'] == ep]
#         # Filter pairs to just the days/stations covered by this episode
#         # (reconstruct the per-event filter to retrieve the daily series)
#         if ep == 'severe_hanoi_haze':
#             ep_pairs = pairs[(pairs['region'] == 'north') & (pairs['season'] == 'dry')
#                               & (pairs['pm25_daily'] > 100)
#                               & (pairs['aod_merged_daily'] > 1.0)]
#         elif ep == 'biomass_burning_MarApr':
#             ep_pairs = pairs[(pairs['month'].isin([3, 4])) & (pairs['pm25_daily'] > 50)]
#         elif ep == 'precip_washout':
#             parts = []
#             for sn, grp in pairs[pairs['season'] == 'wet'].groupby('station_name'):
#                 g = grp.sort_values('date').copy()
#                 g['pm25_delta'] = g['pm25_daily'].diff()
#                 wash = g[g['pm25_delta'] <= -30]['date']
#                 if not wash.empty:
#                     parts.append(g[g['date'].isin(wash)])
#             ep_pairs = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
#         else:
#             ep_pairs = pd.DataFrame()
#         _plot_event(ep, rows, ep_pairs)


## 4. Context — overall §8.1.6 coupling metrics

`pm25_coupling_metrics` (used by the Stage B §8.2 work) is also informative
here: the ALL row tells you whether the held-out merged-AOD-vs-PM2.5
Spearman r aggregated across stations crosses the §1.6 bar even without
filtering to specific events.


In [6]:
# if not pairs.empty:
#     coupling = pm25_coupling_metrics(pairs)
#     if not coupling.empty:
#         cols = ['label', 'aod_type', 'n_stations', 'N',
#                 'pearson_r', 'spearman_r', 'ransac_r2', 'inlier_frac']
#         # ALL row + per-region rows
#         overall = coupling[coupling['label'].isin(['ALL'])
#                             | coupling['label'].str.startswith('region=')]
#         overall[cols].round(3)


## 5. AOD_phys vs PM2.5 — per-station fit on the full held-out window

Section 2 restricted pairs to three episode windows.  This section instead uses
**every** daily `(AOD_phys_daily, pm25_daily)` pair the merged product produced
over `[start, end]` and computes a **RANSAC** fit per station (matching the
`pm25_coupling_metrics` convention in `validate.py` — robust to PM2.5 sensor
glitches and isolated dust/burning spikes that would skew OLS):

* `N` — daily pairs with both values non-NaN
* `pearson_r`, `spearman_r` — linear / rank coupling on all points (unchanged)
* `ransac_r2` — R² on the RANSAC inlier set
* `inlier_frac` — fraction of points the RANSAC consensus kept (sanity check on `ransac_r2`)
* `ransac_slope`, `ransac_intercept` — robust AOD_phys → PM2.5 line, comparable across stations

A summary row aggregates these across stations.  *How* to aggregate is a
design choice (see §5.1).

In [7]:
from scipy import stats
from sklearn.linear_model import RANSACRegressor

def per_station_phys_fit(df_pairs: pd.DataFrame,
                         aod_col: str = 'aod_phys_daily',
                         y_col:   str = 'pm25_daily',
                         min_n:   int = 10) -> pd.DataFrame:
    """One row per station: N, Pearson r, Spearman r, RANSAC R², inlier_frac,
    RANSAC slope/intercept.

    Mirrors `_fit_one` in validate.py (min_samples=0.5, random_state=42) so
    these per-station numbers line up with the ALL row produced by
    `pm25_coupling_metrics`.  `min_n` matches the same 10-pair threshold.
    """
    rows = []
    for sn, grp in df_pairs.groupby('station_name'):
        d = grp[[aod_col, y_col]].dropna()
        if len(d) < min_n:
            continue
        x = d[aod_col].values.reshape(-1, 1)
        y = d[y_col].values
        pr, _ = stats.pearsonr(d[aod_col].values, y)
        sr, _ = stats.spearmanr(d[aod_col].values, y)
        try:
            ransac = RANSACRegressor(min_samples=0.5, random_state=42)
            ransac.fit(x, y)
            inlier_mask      = ransac.inlier_mask_
            ransac_r2        = float(ransac.score(x[inlier_mask], y[inlier_mask]))
            inlier_frac      = float(inlier_mask.mean())
            ransac_slope     = float(ransac.estimator_.coef_[0])
            ransac_intercept = float(ransac.estimator_.intercept_)
        except Exception:
            ransac_r2 = inlier_frac = ransac_slope = ransac_intercept = np.nan
        rows.append({
            'station_name':     sn,
            'region':           grp['region'].iloc[0],
            'N':                len(d),
            'pearson_r':        float(pr),
            'spearman_r':       float(sr),
            'ransac_r2':        ransac_r2,
            'inlier_frac':      inlier_frac,
            'ransac_slope':     ransac_slope,
            'ransac_intercept': ransac_intercept,
        })
    return pd.DataFrame(rows)

if pairs.empty:
    per_st = pd.DataFrame()
    print('No pairs — nothing to fit.')
else:
    per_st = per_station_phys_fit(pairs).sort_values(
        ['region', 'station_name']).reset_index(drop=True)
    print(f'Per-station fits: {len(per_st)} stations '
          f'(min N ≥ 10 daily pairs over the held-out window)')
per_st.round(3)

/home/work1/miniconda3/envs/Airqua_env/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/work1/miniconda3/envs/Airqua_env/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/work1/miniconda3/envs/Airqua_env/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/work1/miniconda3/envs/Airqua_env/lib/python3.12/site-packages/sklearn/metrics/_regression.py:1283: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)
/home/work1/miniconda3/envs/Airqua_env/lib/python3.12/site-packages/skle

Per-station fits: 36 stations (min N ≥ 10 daily pairs over the held-out window)


,station_name,region,N,pearson_r,spearman_r,ransac_r2,inlier_frac,ransac_slope,ransac_intercept
0,Bình Định: Khuôn viên Cây xanh gần cầu chui đư...,central,206,0.015,0.020,0.150,0.510,-11645.010,20.688
1,Gia Lai: KCN Trà Đa - Tp Pleiku (KK),central,135,0.318,0.276,0.911,0.474,82560.218,6.565
2,Gia Lai: UBND thị xã An Khê (KK),central,107,0.193,0.106,0.162,0.505,29601.830,5.127
3,Gia lai: Chư Sê (KK),central,182,0.184,0.160,0.215,0.599,70969.847,0.314
4,Lâm Đồng: Vườn hoa - đối diện THCS Lam Sơn - P...,central,224,0.442,0.292,0.402,0.527,56887.871,11.840
5,Ninh Thuận: Công viên (bến xe cũ) - Đ. Thống N...,central,131,0.219,0.273,0.151,0.504,11525.856,13.855
6,Quảng Nam: Tiếp giáp Đ. Hùng Vương - KDC Đ. Hồ...,central,184,0.456,0.371,0.677,0.576,39944.577,12.850
7,Quảng Ngãi: UBND P. Nguyễn Nghiêm - TP Quảng N...,central,229,0.060,0.061,0.190,0.485,33142.398,21.811
8,Đà Nẵng: Phạm Hùng (KK),central,198,-0.002,-0.143,0.113,0.298,-1378.523,6.808
9,Hà Nam: Công Viên Nam Cao - P.Quang Trung - TP...,north,112,0.467,0.675,0.675,0.652,30746.823,21.108


### 5.1 — Aggregating across stations (your call)

Three defensible answers, with different interpretations:

| Strategy | Formula | What it answers |
|---|---|---|
| **A. Unweighted mean** | `per_st['ransac_r2'].mean()` | "Typical station coupling" — every station counts equally, even the ones with few days. Matches what `pm25_coupling_metrics` ALL row reports (it averages per-station `ransac_r2` the same way). |
| **B. N-weighted mean** | `np.average(per_st['ransac_r2'], weights=per_st['N'])` | "Day-weighted typical R²" — stations with more valid days dominate. Closer to what a naive pooled fit would see. |
| **C. Pooled fit** | one `RANSACRegressor` on *all* `(AOD_phys, PM2.5)` pairs across stations | Treats the whole dataset as one population. Memory note from this project: this collapses because between-station PM2.5 baselines (5–10 µg/m³) dominate the within-station AOD signal. |

The next cell has `aggregate_across_stations(...)`.  My recommendation given the
project memory: **A (unweighted)** for the headline, with **B** alongside it
for transparency.

In [8]:
def aggregate_across_stations(per_st: pd.DataFrame) -> pd.DataFrame:
    """Unweighted mean across stations (strategy A from §5.1).

    Every station counts equally regardless of its N — matches the
    pm25_coupling_metrics ALL row convention.
    """
    if per_st.empty:
        return pd.DataFrame()
    row = {
        'aggregation':      'mean (unweighted)',
        'n_stations':       len(per_st),
        'N':                int(per_st['N'].sum()),
        'pearson_r':        float(per_st['pearson_r'].mean()),
        'spearman_r':       float(per_st['spearman_r'].mean()),
        'ransac_r2':        float(per_st['ransac_r2'].mean()),
        'inlier_frac':      float(per_st['inlier_frac'].mean()),
        'ransac_slope':     float(per_st['ransac_slope'].mean()),
        'ransac_intercept': float(per_st['ransac_intercept'].mean()),
    }
    return pd.DataFrame([row])

if not per_st.empty:
    summary = aggregate_across_stations(per_st)
    display(summary.round(3))

,aggregation,n_stations,N,pearson_r,spearman_r,ransac_r2,inlier_frac,ransac_slope,ransac_intercept
0,mean (unweighted),36,5631,0.259,0.252,0.391,0.538,22865.769,13.934
